In [1]:
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv
import json

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
import os
os.listdir('./models_target/models_cad/custom/')

['bracket_1.obj',
 'doorhandle.obj',
 'eyebolt_m20.obj',
 'logitech_c930e_m.obj',
 'pneumaticfitting_GPC1002.obj',
 'pneumaticfitting_GPC1604.obj',
 'pneumaticfitting_GPY16.obj',
 'steel_connector.obj']

In [3]:
name = 'logitech_c930e_m'

target_mesh_file = f'./models_target/models_cad/custom/{name}.obj'
# target_mesh_file = f'./models_target/models_cad/BOP_ITODD/{name}.ply'

In [4]:
result = gf.compute_best_patch_pairs(
    mesh_path=target_mesh_file,
    mesh_max_triangles = 1500,
    angle_deg=10,              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
    min_opening=1.0,           # 그리퍼 최소 개구(mm)
    max_opening=140.0,         # 그리퍼 최대 개구(mm)
    angle_tolerance_deg=10,    # 패치 페어 정반대 threshold
    top_k=500                  # 상위 패치 페어 후보 개수
)

reports = gf.check_gripper_feasibility_faces_with_yaw(result)
# reports = gf.check_gripper_feasibility_faces_with_rotation(result)

In [5]:
fig = gfv.visualize_merged_patches_plotly(result, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_normal.html", include_plotlyjs="cdn", full_html=True)

In [6]:
fig = gfv.visualize_pairs_centroid_lines(result, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs.html", include_plotlyjs="cdn", full_html=True)

In [7]:
# fig = gfv.visualize_feasible_pairs(result, reports, show=True) # 선만 보기
fig = gfv.visualize_feasible_pairs_pads(result, reports, show=True) # 그리퍼 패드 보기: check_gripper_feasibility_faces_with_yaw
# fig = gfv.visualize_feasible_pairs_with_cylinder(result, reports, show=True, vis_cyl=True) # 실린더 보기: check_gripper_feasibility_faces_with_rotation
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs_feasible.html", include_plotlyjs="cdn", full_html=True)

In [11]:
# --- Epsilon-Quality 계산(점 / 면 접촉) 테스트 ---
report = reports[-12]

result_mesh = result['mesh_patches']
com = result['mesh_quad'].center_mass
patches = {p["id"]: p for p in result["patches"]}

p_i = patches[report['patch_i']]
p_j = patches[report['patch_j']]
f_i = report['face_i']
f_j = report['face_j']

c_i = result_mesh.vertices[result_mesh.faces[f_i]].mean(axis=0)
c_j = result_mesh.vertices[result_mesh.faces[f_j]].mean(axis=0)

n_i = np.asarray(p_i['normal'])
n_j = np.asarray(p_j['normal'])

np.set_printoptions(precision=3, suppress=True, floatmode='fixed')
print(f'ci = {c_i}, cj = {c_j} \nni = {n_i}, nj = {n_j}')

print(report['face_i'], report['face_j'])

ci = [ 36.239   5.225 -16.999], cj = [35.855  5.297 16.998] 
ni = [ 0.002  0.002 -1.000], nj = [0.000 0.000 1.000]
1296 1318


In [12]:
# # 점 접촉 GWS (Force/Torque Space) 시각화 / EQ 계산
# eps_quality = gf.calculate_epsilon_quality(c_i, n_i, c_j, n_j, com, mu=1, k=16)
# print(eps_quality)
# fig_gws = gfv.visualize_wrench_space(c_i, n_i, c_j, n_j, com, mu=1, k=16, show=True)

# 면 접촉 GWS (Force/Torque Space) 시각화 / EQ 계산
eps_quality = gf.calculate_squeeze_epsilon_quality(
    mesh=result['mesh_patches'],
    f_i_idx=p_i['face_indices'],
    f_j_idx=p_j['face_indices'],
    c_i=c_i, n_i=n_i, yaw_i= report['feasible_yaw'],
    c_j=c_j, n_j=n_j, yaw_j=-report['feasible_yaw'],
    com=com,
    squeeze_depth=0.1, # 압착
)
print(f"Squeeze Epsilon Quality: {eps_quality:.6f}")

fig_gws = gfv.visualize_squeeze_wrench_space(
    mesh=result['mesh_patches'],
    f_i_idx=p_i['face_indices'],
    f_j_idx=p_j['face_indices'],
    c_i=c_i, n_i=n_i, yaw_i= report['feasible_yaw'],
    c_j=c_j, n_j=n_j, yaw_j=-report['feasible_yaw'],
    com=com,
    squeeze_depth=0.1, # 압착
    k=8
)

Squeeze Epsilon Quality: 0.149384


In [ ]:
report

## OPE 연계

In [5]:
# SAM6D result 
with open("./RUN_result/output/detection_pem_20251009_023900_logitech_c930e_m.json", "r") as f:
    detections = json.load(f)

# score 가장 높은 detection 선택
best_det = max(detections, key=lambda d: d["score"])
R_oc = np.array(best_det["R"])  # (3x3)
t_oc = np.array(best_det["t"]).reshape(3, 1)  # (3x1)

H_OC = gf.to44(R_oc, t_oc) # C좌표계 기준 O좌표계 원점 위치
H_OC[:3, :3], H_OC[:3, 3]

(array([[ 0.95148522, -0.09229513,  0.29352301],
        [-0.29754043, -0.03298134,  0.95413935],
        [-0.07838156, -0.99518538, -0.05884275]]),
 array([141.44282532, -48.00730133, 431.03399658]))

In [6]:
from scipy.spatial.transform import Rotation

feasible_r = [r for r in reports if r.get('feasible')]
print(f'[GRP] feasible gripping sol.: {len(feasible_r)}')

mesh_patches = result['mesh_patches']
pad_params = gf.PadParams() # 패드 파라미터 미리 로드

feasible_pairs = []
H_dict = {"EE origin": np.eye(4,4)} # EE 좌표계 원점 
# for k in range(min(top_pairs, len(feasible_r))):
for k in range(len(feasible_r)):
    report = reports[k]
    pi = result['patches'][report['patch_i']]
    pj = result['patches'][report['patch_j']]
    ni = gf.unit(np.asarray(pi["normal"], float))
    nj = gf.unit(np.asarray(pj["normal"], float))
    fi = report['face_i']
    fj = report['face_j']
    ci = mesh_patches.vertices[mesh_patches.faces[fi]].mean(axis=0)
    cj = mesh_patches.vertices[mesh_patches.faces[fj]].mean(axis=0)
    yaw_deg = report['feasible_yaw']

    H_OG, stroke = gf.build_gripper_pose_obj_OPE({"centroid": ci, "normal": ni}, {"centroid": cj, "normal": nj}, yaw_deg, H_OC=H_OC)
    H_EdEn, H_OdEn = gf.ee_delta_pose_des(H_OC, H_OG)
    
    r_quat = Rotation.from_matrix(H_EdEn[:3,:3]).as_quat()

    # pair 검사 및 실제로 feasible한 pair 만 남김
    # 그리퍼 접근 각도 45 이하인 경우
    z_H_EdEn = H_EdEn[:3,2] # z축 각도 필터링용
    if np.dot(z_H_EdEn, np.array([0,0,1])) < 0.5: 
        continue
    # # 그리퍼 패드 - 바닥 간섭 검사
    pad_diagonal = np.sqrt(pad_params.pad_w**2 + pad_params.pad_h**2)
    pad_radius = pad_diagonal / 2.0
    H_OdEn_rot = H_OdEn[:3,:3]
    O_rotated = H_OdEn_rot @ mesh_patches.vertices.T
    z_check = O_rotated[-1,:].min() + pad_radius
    ci_world = H_OdEn_rot @ ci
    cj_world = H_OdEn_rot @ cj
    # print(f'{ci[2]:.3f}, {cj[2]:.3f}, {ci_world[2]:.3f}, {cj_world[2]:.3f}')
    if ci_world[2] < z_check or cj_world[2] < z_check:
        continue

    t_quat = H_EdEn[:3,3] / 1000
    res = {"pose_quat": np.concatenate([t_quat, r_quat]),
            "stroke": stroke}
    feasible_pairs.append(res)

    H_dict[f"pair {k+1}"] = H_EdEn

print(f'[GRP] feasible gripping sol.: {len(feasible_pairs)}')

[GRP] feasible gripping sol.: 769
[GRP] feasible gripping sol.: 254


In [7]:
z_check

np.float64(16.515245335907924)

In [8]:
fig = gfv.visualize_frames(H_dict, scale=100, H_OdEn=H_OdEn, result=result)

In [ ]:
yaw_deg

In [ ]:
import os
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv

# for dataset_name in ['custom']: # 'custom' / 'BOP_TLESS' / 'BOP_IPD' / 'BOP_ITODD' / 'BOP_XYZIBD'
for dataset_name in ['custom', 'BOP_ITODD', 'BOP_XYZIBD', 'BOP_IPD', 'BOP_TLESS']:

    dir_path = f'./Grasping_Result (EQ sorted, pad col check, pad rot 0 45 90)/{dataset_name}/'
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
    cads = [file for file in os.listdir(f'./models_target/models_cad/{dataset_name}') if file.endswith('.obj') or file.endswith('.ply')]

    for cad in cads:
        name = cad.split('.')[0]
        fmt  = cad.split('.')[-1]
        target_mesh_file = f'./models_target/models_cad/{dataset_name}/{name}.{fmt}'

        result = gf.compute_best_patch_pairs(
            mesh_path=target_mesh_file,
            mesh_max_triangles = 1500,
            angle_deg=10,              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
            min_opening=1.0,           # 그리퍼 최소 개구(mm)
            max_opening=140.0,         # 그리퍼 최대 개구(mm)
            angle_tolerance_deg=10,    # 패치 페어 정반대 threshold
            top_k=500                  # 상위 패치 페어 후보 개수
        )

        reports = gf.check_gripper_feasibility_faces_with_yaw(result) # yaw (접근방향) 고려 O : Pad Box 회전
        # reports = gf.check_gripper_feasibility_faces_with_rotation(result) # yaw (접근방향) 고려 X : 실린더


        feasible_p = result['top_k']
        feasible_r = [r for r in reports if r.get('feasible')]
        print(f'{dataset_name} - {name} : patch pairs {len(feasible_p)}, feasible sol. {len(feasible_r)}')

        fig = gfv.visualize_merged_patches_plotly(result)
        fig.write_html(f"./{dir_path}/3-patch_normal_{name}.html", include_plotlyjs="cdn", full_html=True)

        try:
            fig = gfv.visualize_pairs_centroid_lines(result)
            fig.write_html(f"./{dir_path}/2-patch_pairs_{name}.html", include_plotlyjs="cdn", full_html=True)
        except:
            print(f'cannot get patch pairs in {dataset_name}:{name}')
            pass

        try:
            # fig = gfv.visualize_feasible_pairs(result, reports) # # yaw (접근방향) 고려 O : 선만 그리기
            fig = gfv.visualize_feasible_pairs_pads(result, reports) # yaw (접근방향) 고려 O : 0, 90도 Pad Box 회전, 패드 그리기
            # fig = gfv.visualize_feasible_pairs_with_cylinder(result, reports, vis_cyl=True) # yaw (접근방향) 고려 X : 실린더
            fig.write_html(f"./{dir_path}/1-grasping_pairs_feasible_{name}_{len(feasible_r)}sol.html", include_plotlyjs="cdn", full_html=True)
        except:
            print(f'no feasible pairs in {dataset_name}:{name}')
            pass

#### Test

In [ ]:
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv
import json

In [ ]:
name = 'logitech_c930e_m' # logitech_c930e_m / obj_000001
target_mesh_file = f'./models_target/models_cad/custom/{name}.obj'

In [ ]:
mesh_path=target_mesh_file
mesh_max_triangles = 1000
angle_deg=15              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
min_opening=1.0           # 그리퍼 최소 개구(mm)
max_opening=140.0         # 그리퍼 최대 개구(mm)
angle_tolerance_deg=15    # 패치 페어 정반대 threshold
top_k=500                  # 상위 패치 페어 후보 개수

In [ ]:
result['top_k'][24]

In [ ]:
mesh_og = trimesh.load(mesh_path)

mesh_filter, mesh_quad = gf.load_uniform_mesh_with_open3d(mesh_path, target_triangles=mesh_max_triangles)

mesh_COM = mesh_quad.center_mass
mesh_bound = float(np.linalg.norm(mesh_quad.bounds[1] - mesh_quad.bounds[0]))
# mesh_filter: (면적 기준 필터링 이후, watertight X), mesh_quad: (면적 기준 필터링 이전, watertight O)
remesh = gf.split_long_edges(mesh_filter)
# remesh = gf.merge_small_faces(remesh)

patches = gf.extract_planar_patches(remesh, angle_deg=angle_deg) # coplanar_tol 삭제
patches = gf.orient_patch_normals(mesh_quad, remesh, patches)  # 모두 바깥쪽으로 정렬

params = gf.PatchPairParams(
    min_opening=min_opening,
    max_opening=max_opening,
    angle_tolerance_deg=angle_tolerance_deg,
)

In [ ]:
gfv.visualize_mesh_with_edges(mesh_og)

In [ ]:
cand = result['top_k'][24]
mesh = result['mesh_patches']
mesh_ch = result['mesh_quad']
patches = result['patches']
com = mesh_ch.center_mass

clearance_out = 10.0
pad_w = 34
pad_h = 21
pad_d = 7

cm = trimesh.collision.CollisionManager()
cm.add_object("part", mesh_ch) # 간섭 검사용: 면적 필터 안 된 메시 

In [ ]:
pid_i, pid_j = cand["patch_i"], cand["patch_j"]

p_i, p_j = patches[pid_i], patches[pid_j]
n_i = gf.unit(np.asarray(p_i["normal"], float))
n_j = gf.unit(np.asarray(p_j["normal"], float))

# pair별로 가능한 모든 후보를 찾기
reports_for_this_pair = []
mesh_F, mesh_V = mesh.faces, mesh.vertices
for f_i in p_i["face_indices"]:
    ci = mesh_V[mesh_F[f_i]].mean(axis=0)

    best_j   = None
    best_cj  = None
    best_scr = -1.0
    for f_j in p_j["face_indices"]:
        cj = mesh_V[mesh_F[f_j]].mean(axis=0)
        d  = cj - ci
        nd = np.linalg.norm(d)
        if nd < 1e-12:
            continue
        d_hat = d / nd
        scr = abs(float(d_hat @ (-n_i))) * abs(float((-d_hat) @ n_j))
        if scr > best_scr:
            best_scr, best_j, best_cj = scr, f_j, cj
    if best_j is None:
        continue
    # 엇갈린 face pair 건너뛰기
    alignment_dist_i = gf.point_line_distance(ci, -n_i, best_cj)
    alignment_dist_j = gf.point_line_distance(best_cj, -n_j, ci)
    # dist_criteria = min(pad_w, pad_h) / 5 # 엇갈림 기준
    dist_criteria = 10 # 엇갈림 기준
    if alignment_dist_i > dist_criteria or alignment_dist_j > dist_criteria:
        print(f'alignment error, {f_i} {f_j} ')


    yaw=90
    box_i = gf.make_rot_pad_box_at_patch(
        {"centroid": ci, "normal": n_i},
        pad_w, pad_h, pad_d, clearance_out, yaw_deg=yaw
    )
    box_j = gf.make_rot_pad_box_at_patch(
        {"centroid": best_cj, "normal": n_j},
        pad_w, pad_h, pad_d, clearance_out, yaw_deg=-yaw
    )
    # 충돌이 발생하면 이 yaw는 건너뛰고 다음 yaw를 검사.
    if cm.in_collision_single(box_i): print(f'collision, {f_i} {f_j}')
    if cm.in_collision_single(box_j): print(f'collision, {f_i} {f_j}')

                    
    midpoint = 0.5 * (ci + best_cj)
    current_dist = np.linalg.norm(midpoint - com)
    
    Fi = -n_i; Fj = -n_j
    tau = np.cross(ci - com, Fi) + np.cross(best_cj - com, Fj)
    current_moment = float(np.linalg.norm(tau))